In [1]:
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
builder = (SparkSession.builder
           .appName("data-skipping-delta-table")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "2g")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8498b32d-ebcf-44a1-8724-93e1be501415;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 196ms :: artifacts dl 10ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0 

In [3]:
%load_ext sparksql_magic
%config SparkSql.limit=20

In [4]:
# Create some sample data frames
# A large data frame with 1 million rows
df = (spark.range(0, 1000000)
            .withColumn("salary", 100*(rand() * 100).cast("int"))
            .withColumn("gender", when((rand() * 2).cast("int") == 0, "M").otherwise("F"))
            .withColumn("country_code", 
                        when((rand() * 4).cast("int") == 0, "US")
                        .when((rand() * 4).cast("int") == 1, "CN")
                        .when((rand() * 4).cast("int") == 2, "IN")
                        .when((rand() * 4).cast("int") == 3, "BR")
                        .otherwise('RU')))
df.show(5)

+---+------+------+------------+
| id|salary|gender|country_code|
+---+------+------+------------+
|  0|  1600|     M|          US|
|  1|   200|     M|          BR|
|  2|  2400|     F|          RU|
|  3|   800|     F|          RU|
|  4|  2700|     F|          US|
+---+------+------+------------+
only showing top 5 rows



In [5]:
(df.write
 .format("delta")
 .mode("overwrite")
 .save("../data/tmp/employee_salary"))

In [6]:
df = (spark.range(0, 1000)
            .withColumn("salary", 100*(rand() * 100).cast("int"))
            .withColumn("gender", when((rand() * 2).cast("int") == 0, "M").otherwise("F"))
            .withColumn("country_code", 
                        when((rand() * 4).cast("int") == 0, "US")
                        .when((rand() * 4).cast("int") == 1, "CN")
                        .when((rand() * 4).cast("int") == 2, "IN")
                        .when((rand() * 4).cast("int") == 3, "BR")
                        .otherwise('RU')))
(df.write
 .format("delta")
 .mode("append")
 .save("../data/tmp/employee_salary"))

In [7]:
%%sparksql
DESCRIBE HISTORY delta.`/opt/workspace/data/tmp/employee_salary`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2025-03-01 21:49:51.644000,null,null,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",null,null,null,0,Serializable,True,"{'numOutputRows': '1000', 'numOutputBytes': '12904', 'numFiles': '4'}",null,Apache-Spark/3.4.1 Delta-Lake/2.4.0
0,2025-03-01 21:49:30.793000,null,null,WRITE,"{'mode': 'Overwrite', 'partitionBy': '[]'}",null,null,null,null,Serializable,False,"{'numOutputRows': '1000000', 'numOutputBytes': '5402060', 'numFiles': '2'}",null,Apache-Spark/3.4.1 Delta-Lake/2.4.0


In [8]:
%%sparksql
OPTIMIZE delta.`/opt/workspace/data/tmp/employee_salary`

path,metrics
file:/opt/workspace/data/tmp/employee_salary,"Row(numFilesAdded=1, numFilesRemoved=6, filesAdded=Row(min=5406889, max=5406889, avg=5406889.0, totalFiles=1, totalSize=5406889), filesRemoved=Row(min=3038, max=2701401, avg=902494.0, totalFiles=6, totalSize=5414964), partitionsOptimized=1, zOrderStats=None, numBatches=1, totalConsideredFiles=6, totalFilesSkipped=0, preserveInsertionOrder=False, numFilesSkippedToReduceWriteAmplification=0, numBytesSkippedToReduceWriteAmplification=0, startTimeMs=1740865820113, endTimeMs=0, totalClusterParallelism=4, totalScheduledTasks=0, autoCompactParallelismStats=None, deletionVectorStats=None, numTableColumns=4, numTableColumnsWithStats=4)"


In [9]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

In [10]:
%%sparksql
VACUUM delta.`/opt/workspace/data/tmp/employee_salary` RETAIN 0 HOURS

Deleted 6 files and directories in a total of 1 directories.


path
file:/opt/workspace/data/tmp/employee_salary


In [22]:
spark.stop()